In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
(data_train, data_val, data_test), data_info = tfds.load('cats_vs_dogs',
                                               split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
                                               shuffle_files=True,
                                               as_supervised=True,
                                               with_info=True
                                               )

In [ ]:
print("Traing set size: ", len(data_train))
print("Validation set size: ", len(data_val))
print("Test set size: ", len(data_test))

In [ ]:
def normalize_img(image, label):
    image = tf.image.resize(image, (128, 128))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
# Training pipeline
data_train = data_train.map(normalize_img)
# data_train = data_train.cache()
data_train = data_train.shuffle(1000)
data_train = data_train.batch(32)
data_train = data_train.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Validation pipeline
data_val = data_val.map(normalize_img)
data_val = data_val.batch(32)
# data_val = data_val.cache()
data_val = data_val.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Testing pipeline
data_test = data_test.map(normalize_img)
data_test = data_test.batch(32)
# data_test = data_test.cache()

In [ ]:
# Flat sequential model

model = keras.models.Sequential([
    layers.Input(shape = (128, 128, 3)),
    layers.Flatten(),
    layers.Dense(10, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss = keras.losses.binary_crossentropy,
              metrics = [keras.metrics.BinaryAccuracy()])

In [ ]:
model.fit(data_train, epochs=10, validation_data=data_val)

In [ ]:
print(model.history.history.keys())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (12, 4))

axes[0].plot(model.history.history["loss"], label = "Training loss")
axes[0].plot(model.history.history["val_loss"], label = "Validation loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(model.history.history['binary_accuracy'], label = "Training Accuracy")
axes[1].plot(model.history.history['val_binary_accuracy'], label = "Validation Accuracy")
axes[1].set_title('Accuracy')
axes[1].legend()

plt.show()